### The Architecture: How Multimodal RAG Works

Since standard embedding models (like `nomic-embed-text`) only understand text, we can't embed images directly into the same vector space easily. The industry-standard workaround is:

1. **Extract**: Parse the PDF to separate raw text, tables, and images.
2. **Summarize**: Use a Vision LLM to generate a text summary of every image and table.
3. **Embed**: Embed these text summaries (along with standard text chunks) into ChromaDB.
4. **Retrieve & Answer**: When a user asks a question, retrieve the most relevant text chunks or summaries, and pass them to the LLM to generate an answer.

## 0. Local Infrastructure Setup

To ensure absolute compliance with enterprise data privacy standards and eliminate execution costs, this entire curriculum operates locally using **Ollama** running an optimized `llama3` instance. Run the following cells to initialize your local environment.

In [1]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [2]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

### Environment Setup & Pulling the Vision Model

To process images, we need to add a Multimodal LLM to your Ollama setup. We will use **LLaVA** (Large Language-and-Vision Assistant). We also need the `unstructured` library to parse PDFs into separate text, table, and image elements.

In [4]:
# 1. Install required libraries for PDF parsing and Multimodal RAG
!pip install -qU unstructured[pdf] pdf2image poppler-utils tesseract-ocr
!pip install -qU langchain-chroma langchain-core

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tesseract-ocr
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tesseract-ocr)


In [5]:
# 2. Pull the Vision Model via Ollama
!ollama pull llava
!ollama pull llama3
!ollama pull nomic-embed-text

### Document Parsing (Extracting Text, Tables, and Images)

We will use the `unstructured` library to slice an invoice into its structural components.

In [6]:
# Install system dependencies for PDF and OCR processing
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr

# Install the Python libraries
!pip install -qU "unstructured[pdf]" pdf2image pytesseract langchain-chroma langchain-core

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the ne

In [7]:
from unstructured.partition.pdf import partition_pdf

# We assume you have a scanned invoice uploaded as 'sample_invoice.pdf'
# pdf_path = "/content/sample_invoice.pdf"
pdf_path = "/content/Invoice.pdf"

print("Partitioning document into Text, Tables, and Images...")
# Extract elements. It saves extracted images to the specified directory.
raw_pdf_elements = partition_pdf(
    filename=pdf_path,
    extract_images_in_pdf=True,
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=4000,
    new_after_n_chars=3800,
    combine_text_under_n_chars=2000,
    image_output_dir_path="/content/invoice_images/"
)

Partitioning document into Text, Tables, and Images...


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [8]:
# Categorize the extracted elements
texts = []
tables = []

for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        tables.append(str(element))
    elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
        texts.append(str(element))

print(f"Extracted {len(texts)} text chunks and {len(tables)} tables.")

Extracted 6 text chunks and 5 tables.


### Summarizing Images and Tables with LLaVA

Now, we use Ollama's `llava` model to look at the extracted tables and images and generate descriptive text summaries.

In [9]:
!pip install -qU langchain-ollama

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
import base64
import os

# Initialize LLaVA for vision tasks
llava_chat = ChatOllama(model="llava", base_url="http://localhost:11434", temperature=0.1)

In [11]:
def summarize_image(image_path):
    """Encodes an image to base64 and asks LLaVA to summarize it."""
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

    prompt = "You are an AI assistant. Detail the contents of this invoice image, including company names, totals, and line items."

    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encoded_string}"}}
        ]
    )
    response = llava_chat.invoke([message])
    return response.content

In [12]:
# 1. Summarize all extracted images
image_summaries = []
image_dir = "/content/invoice_images/"
if os.path.exists(image_dir):
    for img_file in os.listdir(image_dir):
        if img_file.endswith(('.png', '.jpg', '.jpeg')):
            summary = summarize_image(os.path.join(image_dir, img_file))
            image_summaries.append(summary)

In [13]:
# 2. Summarize tables (using llama3 since tables are already extracted as text strings)
llama3_chat = ChatOllama(model="llama3", base_url="http://localhost:11434", temperature=0.1)

table_summaries = []
for table in tables:
    prompt = f"Summarize the following table from an invoice, highlighting key numbers and totals:\n{table}"
    summary = llama3_chat.invoke([HumanMessage(content=prompt)]).content
    table_summaries.append(summary)

### Storing Summaries in ChromaDB

We now treat the image summaries, table summaries, and raw text chunks as our context pool and embed them into ChromaDB using `nomic-embed-text`.

In [14]:
!pip install -qU langchain-community langchain-chroma langchain-core

In [15]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

/tmp/ipykernel_38781/1758688411.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings


In [16]:
# 1. Initialize Embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

/tmp/ipykernel_38781/3644190250.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")


In [17]:
# 2. Package everything into Document objects
doc_contents = texts + table_summaries + image_summaries
documents = [Document(page_content=text, metadata={"source": "Invoice_Case_Study"}) for text in doc_contents]

In [18]:
# 3. Build the Vector Store
print("Embedding multimodal summaries into ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./invoice_multimodal_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector database ready!")

Embedding multimodal summaries into ChromaDB...
Vector database ready!


### The Invoice Q&A Pipeline

Finally, we connect the retriever to `llama3` to answer user questions based on the multimodal summaries.

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Build the Q&A Prompt
template = """You are an AI financial assistant analyzing company invoices.
Answer the question based ONLY on the following context (which includes text, and summaries of tables and images).

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG Chain
multimodal_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llama3_chat
    | StrOutputParser()
)

### Testing

In [21]:
def query_invoice(question):
    print(f"User: {question}")
    response = multimodal_rag_chain.invoke(question)
    print(f"AI Assistant: {response}\n" + "="*50)

In [22]:
# Test Cases for the Invoice
# query_invoice("What is the total amount due on this invoice?")
query_invoice("Which training module had the highest unit rate?")

User: Which training module had the highest unit rate?
AI Assistant: Based on the provided context, specifically the "Itemised Charges" table, I can answer your question.

The training modules with their corresponding unit rates are:

1. Data Analytics
2. Python Full Stack
3. Generative AI Fundamentals
4. Agentic AI with LangChain

Unfortunately, the table does not provide a clear ranking of the training modules by their unit rates. However, I can tell you that the highest unit rate is INR 7 Lakh Twenty-Four Thousand One Hundred and Sixty-Six Only (which is likely an aggregate amount for all services provided).

If you're looking for more specific information about individual module costs, I'd be happy to help with any further questions!


In [23]:
# query_invoice("List all the line items and their individual costs.")
query_invoice("What does the revenue trend chart indicate about May 2025?")

User: What does the revenue trend chart indicate about May 2025?
AI Assistant: According to the context, the revenue trend chart indicates that May 2025 marked the peak engagement month driven by the AI/ML fast-track batch.


In [24]:
# query_invoice("What is the name of the vendor who issued this invoice?")
query_invoice("What are the GST details applied on this invoice?")

User: What are the GST details applied on this invoice?
AI Assistant: According to the context, the GST details applied on this invoice are:

* **CGST (9%):** ₹55,233
* **SGST (9%):** ₹55,233
